In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils import data
from torch import nn
import matplotlib.pyplot as plt
batch_size=256
lr=0.1
epoch=20
x=torch.arange(8.).reshape(2,4)
print(x)
y=x.mean(dim=0)
print(y)
print(y.size())

tensor([[0., 1., 2., 3.],
        [4., 5., 6., 7.]])
tensor([2., 3., 4., 5.])
torch.Size([4])


In [ ]:
def batch_norm(x,gamma,beta,moving_mean,moving_var,momentum,eps):
    if not torch.is_grad_enable():
        x_hat=(x-moving_mean)/torch.sqrt(moving_var+eps)
    else:
        if(len(x.shape)==2):
            x_mean=x.mean(dim=0)
            x_var=((x-x_mean)**2).mean(dim=0)
        else:
            x_mean=x.mean(dim=(0,2,3),keepdim=True)
            x_var=((x - x_mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)
        x_hat = (x - x_mean) / torch.sqrt(x_var + eps)
        y=gamma*x+beta
        moving_mean=momentum*moving_mean+(1-momentum)*x_mean
        moving_var=momentum*moving_var+(1-momentum)*x_var
        return y,moving_mean.data,moving_var.data



In [ ]:
class batchnorm(nn.Module):
    def __init__(self,num_features,num_dim):
        super().__init__()
        if(num_dim==2):
            shape=(1,num_features)
        else:
            shape=(1,num_features,1,1)
        self.gamma=nn.parameter(torch.ones(shape))
        self.beta=nn.parameter(torch.zeros(shape))
        self.moving_mean=torch.zeros(shape)
        self.moving_var=torch.ones(shape)

    def forward(self,x):
        if self.moving_mean.device != x.device:
            self.moving_mean = self.moving_mean.to(x.device)
            self.moving_var = self.moving_var.to(x.device)
        y,self.moving_mean,self.moving_var=batch_norm(x,self.gamma,self.beta,self.moving_mean,self.moving_var,0.9,1e-5)
        return y




In [2]:
def data_maker():
    transform = transforms.ToTensor()
    """
    transform = transforms.Compose([
        transforms.Resize((12, 12)),
        transforms.ToTensor()
    ])
    """
    
    train_dataset = datasets.FashionMNIST(
        root='../data',
        train=True,
        transform=transform,
        download=True
    )
    
    test_dataset = datasets.FashionMNIST(
        root='../data',
        train=False,
        transform=transform,
        download=True
    )
    return train_dataset,test_dataset


In [3]:
def data_loader(data_array,batch_size,flag):
    return data.DataLoader(data_array,batch_size,shuffle=flag)
train_dataset,test_dataset=data_maker()
train_iter=data_loader(train_dataset,batch_size,True)
test_iter=data_loader(test_dataset,batch_size,False)




In [4]:
def net_maker():
    net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.BatchNorm2d(6, momentum=0.1),nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.BatchNorm2d(16, momentum=0.1),nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.BatchNorm1d(120, momentum=0.1),nn.Sigmoid(),
    nn.Linear(120, 84), nn.BatchNorm1d(84, momentum=0.1),nn.Sigmoid(),
    nn.Linear(84, 10))
    return net

In [5]:
def init(m):
    if isinstance(m,nn.Linear) or isinstance(m,nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)

In [6]:
net=net_maker().to('cuda')
net.apply(init)

loss=nn.CrossEntropyLoss()

trainer=torch.optim.SGD(net.parameters(),lr=lr)



In [7]:
import copy


def evaluate_current_mode(model, data_iter):
    total_loss = 0.0
    total_acc = 0
    total_count = 0

    with torch.no_grad():
        for x, y in data_iter:
            x = x.to("cuda")
            y = y.to("cuda")

            logits = model(x)
            l = loss(logits, y)

            total_loss += l.item() * y.size(0)
            total_acc += (
                logits.argmax(dim=1) == y
            ).sum().item()
            total_count += y.size(0)

    return (
        total_loss / total_count,
        total_acc / total_count
    )

In [8]:
for i in range(epoch):
    net.train()
    optimize_loss = 0.0
    optimize_count = 0
    for x,y in train_iter:
        x=x.to("cuda")
        y=y.to("cuda")
        trainer.zero_grad()
        logit=net(x)
        l=loss(logit,y)
        l.backward()
        trainer.step()
        optimize_loss += l.item() * y.size(0)
        optimize_count += y.size(0)

    print(
        f"第{i+1}轮训练过程loss："
        f"{optimize_loss / optimize_count:.5f}"
    )
    net.eval()
    running_loss, running_acc = evaluate_current_mode(
        net,
        test_iter
    )

    # 复制相同参数，但使用当前batch统计量
    probe_net = copy.deepcopy(net)
    probe_net.train()

    batch_loss, batch_acc = evaluate_current_mode(
        probe_net,
        test_iter
    )

    del probe_net

    print(
        f"running统计：loss={running_loss:.5f}, "
        f"acc={running_acc:.5f}"
    )

    print(
        f"batch统计：loss={batch_loss:.5f}, "
        f"acc={batch_acc:.5f}"
    )
    train_acc=0
    test_acc=0
    train_loss=0.0
    test_loss=0.0
    train_count=0
    test_count=0
    with torch.no_grad():
        for x,y in train_iter:
            x=x.to("cuda")
            y=y.to("cuda")
            logit=net(x)
            train_count+=y.size(0)
            l=loss(logit,y)
            train_loss+=l*y.size(0)
            train_acc+=(logit.argmax(dim=1)==y).sum().item()
        print(f"第{i+1}轮的训练loss：{train_loss/train_count:.5f},训练准确率：{train_acc/train_count}")
        for x,y in test_iter:
            x=x.to("cuda")
            y=y.to("cuda")
            logit=net(x)
            test_count+=y.size(0)
            l=loss(logit,y)
            test_loss+=l*y.size(0)
            test_acc+=(logit.argmax(dim=1)==y).sum().item()
        print(f"第{i+1}轮的测试loss：{test_loss/test_count:.5f},测试准确率：{test_acc/test_count}")



第1轮训练过程loss：0.89439
running统计：loss=0.74893, acc=0.74130
batch统计：loss=0.66778, acc=0.77940
第1轮的训练loss：0.72514,训练准确率：0.7553666666666666
第1轮的测试loss：0.74893,测试准确率：0.7413
第2轮训练过程loss：0.55296
running统计：loss=0.54539, acc=0.81870
batch统计：loss=0.52470, acc=0.82310
第2轮的训练loss：0.51919,训练准确率：0.83195
第2轮的测试loss：0.54539,测试准确率：0.8187
第3轮训练过程loss：0.46848
running统计：loss=0.67445, acc=0.76380
batch统计：loss=0.53789, acc=0.80840
第3轮的训练loss：0.64971,训练准确率：0.7678833333333334
第3轮的测试loss：0.67445,测试准确率：0.7638
第4轮训练过程loss：0.42321
running统计：loss=0.46091, acc=0.84580
batch统计：loss=0.43238, acc=0.84960
第4轮的训练loss：0.42925,训练准确率：0.8564666666666667
第4轮的测试loss：0.46091,测试准确率：0.8458
第5轮训练过程loss：0.39243
running统计：loss=0.64846, acc=0.75770
batch统计：loss=0.44865, acc=0.84820
第5轮的训练loss：0.62063,训练准确率：0.7672666666666667
第5轮的测试loss：0.64846,测试准确率：0.7577
第6轮训练过程loss：0.36827
running统计：loss=0.52843, acc=0.80420
batch统计：loss=0.44064, acc=0.84020
第6轮的训练loss：0.49087,训练准确率：0.8196
第6轮的测试loss：0.52843,测试准确率：0.8042
第7轮训练过程loss：0.35280
running

在上述实验中，我们发现，BatchNorm 的 running_mean 和 running_var 统计不准确时，会导致模型在 net.eval() 模式下的推理结果不稳定，并与训练模式下使用当前批次统计量得到的结果存在明显差异。
因此，我们先正常完成模型训练，再固定网络权重以及 BatchNorm 的 γ、β 参数，使用整个训练集重新进行一次前向传播。这个过程不计算梯度，也不更新模型参数，只采用累积平均的方式重新统计各 BatchNorm 层的 running_mean 和 running_var，使其更准确地反映最终模型产生的特征分布，从而改善模型在推理模式下的表现。
BatchNorm 的 running_mean 和 running_var 会受到近期批次及其数据分布的影响，因此它们可能无法准确反映最终模型的特征分布。这会导致模型在 net.eval() 模式下的结果出现波动。
实际上参数是一直越训越好，就是遇到某些批次被影响导致的，所以只要最后一遍重新统计即可

In [9]:
from torch.optim.swa_utils import update_bn

device = torch.device("cuda")

# 专门用于重新统计BatchNorm的数据迭代器
bn_iter = data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True
)

# 固定已经训练好的模型参数，
# 使用整个训练集重新计算running_mean和running_var
update_bn(
    bn_iter,
    net,
    device=device
)

# 重新统计结束后切换到预测模式
net.eval()

Sequential(
  (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): Sigmoid()
  (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (4): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (5): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): Sigmoid()
  (7): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (8): Flatten(start_dim=1, end_dim=-1)
  (9): Linear(in_features=400, out_features=120, bias=True)
  (10): BatchNorm1d(120, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): Sigmoid()
  (12): Linear(in_features=120, out_features=84, bias=True)
  (13): BatchNorm1d(84, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (14): Sigmoid()
  (15): Linear(in_features=84, out_features=10, bias=True)
)

In [10]:
test_loss = 0.0
test_acc = 0
test_count = 0

with torch.no_grad():
    for x, y in test_iter:
        x = x.to(device)
        y = y.to(device)

        logits = net(x)
        l = loss(logits, y)

        test_loss += l.item() * y.size(0)
        test_acc += (
            logits.argmax(dim=1) == y
        ).sum().item()
        test_count += y.size(0)

print(
    f"重新统计后的测试loss："
    f"{test_loss / test_count:.5f}, "
    f"测试准确率："
    f"{test_acc / test_count:.5f}"
)

重新统计后的测试loss：0.31292, 测试准确率：0.88380
